<h1 align="center">Detecção Automática do Uso de EPI em Ambiente Simples com Visão Computacional e Aprendizado Profundo</h1>

<p align="center">
<b>Lucas Rodrigues Teixeira, Pedro Henrique Garcez Silva, Roberto Sene Azevedo</b><br>
Equipe <i>Ctrl+C, Ctrl+V e Fé</i><br>
Universidade Federal do ABC (UFABC) — ESZA019 Visão Computacional — 2026.2<br>
Prof. Celso Setsuo Kurashima<br>
<code>{lucas, pedro, roberto}@aluno.ufabc.edu.br</code>
</p>

---


**Resumo.** *A fiscalização do uso de Equipamentos de Proteção Individual (EPI) é, em geral, manual e
intermitente, sujeita a falhas e sem registro objetivo. Este trabalho apresenta um sistema de visão
computacional em tempo real que verifica automaticamente o uso de capacete e colete de alta visibilidade na
entrada de uma área controlada, sinalizando conformidade. A solução combina calibração de câmera
(erro de reprojeção de 0,127 px), um detector de objetos baseado em rede neural convolucional (YOLOv8),
associação espacial EPI–pessoa e uma decisão temporal estabilizada. Em CPU, o sistema opera a
aproximadamente 8 quadros por segundo. A validação com oito voluntários resultou em pontuação SUS média de
83,75 (95,0 desconsiderando duas respostas atípicas), indicando boa usabilidade. Os resultados demonstram a
viabilidade de um verificador de EPI de baixo custo em ambiente controlado.*

**Palavras-chave:** visão computacional, detecção de objetos, EPI, YOLOv8, segurança do trabalho,
calibração de câmera.

**Abstract.** *Compliance checking of Personal Protective Equipment (PPE) is usually manual and
intermittent, error-prone and without objective records. This work presents a real-time computer vision
system that automatically verifies the use of a hard hat and a high-visibility vest at the entrance of a
controlled area. The solution combines camera calibration (0.127 px reprojection error), a CNN-based object
detector (YOLOv8), spatial PPE–person association and a temporally stabilized decision. On CPU it runs at
about 8 FPS. A usability study with eight volunteers yielded a mean SUS score of 83.75 (95.0 excluding two
outlier responses), indicating good usability.*

## 1. Introdução

Equipamentos de Proteção Individual (EPI) — capacete, colete refletivo, óculos, luvas — constituem a
primeira barreira contra acidentes em canteiros, oficinas e laboratórios. Apesar da obrigatoriedade legal,
a verificação do **uso correto** ainda é feita, na maioria dos casos, de forma **manual e intermitente** por
um responsável de segurança: um processo sujeito a falha humana, que não cobre todos os instantes e não
gera registro objetivo. A partir de entrevistas empáticas conduzidas na fase inicial do projeto, a equipe
identificou essa lacuna como uma dor real e propôs automatizá-la com **visão computacional**.

Este artigo descreve um sistema que, a partir de uma câmera fixa na entrada de uma área controlada,
verifica **em tempo real** se a pessoa que se aproxima usa os EPIs obrigatórios (capacete e colete) e
sinaliza **CONFORME** ou **NÃO CONFORME**. Para manter o escopo específico e preciso, adota-se um
**ambiente simples**: uma pessoa por vez, iluminação e fundo controlados e câmera fixa previamente
calibrada. Sobre esse cenário, emprega-se **aprendizado profundo** (um detector de objetos de estágio
único) para localizar, no mesmo quadro, a pessoa e os EPIs, decidindo a conformidade por **associação
espacial** e **estabilização temporal**.

As principais contribuições são: (i) um *pipeline* completo e reprodutível de verificação de EPI que integra
**calibração de câmera** (Requisito do trabalho) e **detecção por CNN**; (ii) uma avaliação
**multidimensional** (qualidade da calibração, desempenho em tempo real e usabilidade com usuários reais);
e (iii) a discussão de limitações e caminhos de evolução para uso prático.

## 2. Trabalhos relacionados

A **detecção de objetos** evoluiu de métodos clássicos (descritores manuais como HOG/SIFT + classificadores)
para **redes neurais convolucionais**. Detectores de **estágio único**, como a família **YOLO** [Redmon et
al. 2016] e o **SSD**, preveem caixas e classes numa única passagem, oferecendo excelente compromisso entre
acurácia e velocidade — adequado a aplicações em tempo real. O **YOLOv8** [Jocher et al. 2023] é uma
implementação moderna e leve, com variantes *nano/small* que rodam em CPU.

A verificação de EPI por visão é um tema recorrente em segurança do trabalho, com diversos conjuntos de
dados públicos (por exemplo, *Hard Hat Workers* e *Construction Site Safety*) que anotam pessoas e
equipamentos. A **calibração de câmera** pelo método do tabuleiro [Zhang 2000] é padrão para corrigir a
distorção da lente antes de qualquer medição geométrica, e foi trabalhada nos Laboratórios 4 e 5 da
disciplina. Este trabalho reúne essas peças em um sistema fim-a-fim voltado ao cenário de "ambiente
simples".

## 3. Materiais e métodos

### 3.1 Visão geral do sistema
O *pipeline* de tempo real executa, para cada quadro:

```
[Câmera USB] → [Calibração / undistort] → [YOLOv8 (CNN) + rastreamento]
   → [Associação EPI↔pessoa (IoU/contenção)] → [Decisão temporal (N quadros)]
   → [Status CONFORME/NÃO CONFORME + FPS + gravação]
```

### 3.2 Calibração da câmera
A câmera foi calibrada pelo método do tabuleiro de xadrez [Zhang 2000] com 15 imagens, obtendo a matriz
intrínseca $K$ e os coeficientes de distorção. Cada quadro é corrigido (`cv2.undistort`) **antes** da
detecção, evitando que a distorção deforme as caixas e a associação. O **erro de reprojeção** obtido foi de
**0,127 px**, indicando calibração de alta qualidade.

### 3.3 Detector e treinamento
Utilizou-se **transfer learning** a partir do **YOLOv8n** pré-treinado, adaptado para três classes —
**pessoa**, **capacete** e **colete** — em um conjunto público de EPI (Construction Site Safety, Roboflow),
treinado no Google Colab com GPU. O detector aplica limiar de confiança e *non-maximum suppression* (NMS).

### 3.4 Associação e decisão de conformidade
Cada capacete/colete é vinculado à pessoa cuja caixa o contém (sobreposição/IoU: capacete na região da
cabeça, colete no tronco). A regra lógica é
$$\text{CONFORME} \iff (\text{pessoa}) \wedge (\text{capacete}) \wedge (\text{colete}).$$
A decisão só é confirmada após **N quadros** consecutivos coerentes (estabilização temporal por rastreamento),
evitando oscilação do rótulo. Prioriza-se **alta revocação** para a classe *NÃO CONFORME* (é preferível um
alarme falso a liberar alguém sem EPI).

### 3.5 Protocolo de avaliação
Avaliou-se o sistema em três frentes: **(a)** qualidade da **calibração** (erro de reprojeção);
**(b)** **desempenho** em tempo real (FPS/latência); **(c)** **detecção/decisão** (mAP do detector e matriz
de confusão da decisão, via `avaliar_metricas.py`); e **(d)** **usabilidade** com voluntários (escala SUS),
seguindo o roteiro de testes da Etapa 5.

## 4. Resultados e discussão

### 4.1 Calibração
A calibração convergiu com **erro de reprojeção de 0,127 px** (15 imagens), bem abaixo de 1 px — a geometria
corrigida não introduz erro relevante na detecção nem na associação.

### 4.2 Desempenho em tempo real
O sistema operou a **≈ 8 FPS em CPU** (latência ~125 ms/quadro), suficiente para o cenário de uma pessoa
entrando por vez. A Figura 1 mostra um quadro real da sessão gravada, com o status **NÃO CONFORME** e o
rótulo **"sem capacete"** a 8,2 FPS.

![Figura 1 — quadro da sessão](codigos/frame_sessao.png)
<p align="center"><i>Figura 1. Quadro real do sistema em operação (sessao_epi.mp4).</i></p>

### 4.3 Detecção e decisão
O detector reconheceu pessoa, capacete e colete nas situações de teste (com/sem cada EPI), conforme
confirmado qualitativamente pelos voluntários. As métricas quantitativas de **mAP** e a **matriz de
confusão** da decisão devem ser preenchidas com a execução do `avaliar_metricas.py` sobre o conjunto
rotulado; nesta versão permanecem *a preencher*, por não terem sido medidas — opção adotada para preservar a
integridade dos resultados (Portaria CNPq 2664/2026).

### 4.4 Usabilidade
Na validação com **8 voluntários** (10/08/2026), a pontuação **SUS média foi 83,75** (95,0 excluindo duas
respostas atípicas com padrão "tudo 5"), acima do patamar de 68 que caracteriza boa usabilidade. **Todos**
os participantes descreveram corretamente o objetivo do sistema sem instrução prévia, evidenciando interface
autoexplicativa. Os elogios recaíram sobre a **rapidez** e a **facilidade de uso**; as sugestões — mais
velocidade, **interface gráfica** e **avaliação parcial** (indicar qual EPI falta) — são tecnicamente
viáveis com a arquitetura atual.

![Figura 2 — SUS por voluntário](Trabalho Final/analise_sus_respondentes.png)
<p align="center"><i>Figura 2. Pontuação SUS por voluntário (referência: 68 = bom).</i></p>

### 4.5 Limitações
O escopo "ambiente simples" (uma pessoa por vez, iluminação/fundo controlados) delimita conscientemente o
problema; cenas com múltiplas pessoas, iluminação adversa ou fundo complexo degradam a detecção. O modelo,
treinado com dados públicos, pode não capturar perfeitamente a cor exata do colete/capacete da equipe,
o que se resolve com *fine-tuning* usando imagens próprias.

## 5. Conclusão

Apresentou-se um sistema de **verificação automática do uso de EPI** em tempo real que integra calibração de
câmera, detecção por CNN (YOLOv8), associação espacial e decisão temporal, no escopo de ambiente simples. Os
resultados — **erro de reprojeção de 0,127 px**, **≈ 8 FPS em CPU** e **SUS de 83,75 (95,0 sem atípicos)** —
demonstram a **viabilidade** da proposta e sua **boa usabilidade**. Como trabalhos futuros, destacam-se o
*fine-tuning* com imagens próprias, a medição do mAP e da matriz de confusão sobre um conjunto rotulado, uma
interface gráfica com avaliação parcial de EPI, o suporte a múltiplas pessoas e a otimização de desempenho
(GPU/modelo mais leve). O uso de câmera estéreo (Labs 5–6) surge como extensão natural para restringir a
verificação à zona monitorada por profundidade.

---
> **Declaração de uso de IA Generativa (Portaria CNPq nº 2664/2026).** Este artigo teve **apoio da
> ferramenta Claude (Anthropic)** na redação e na estruturação do texto. Os dados experimentais foram
> produzidos pela equipe; os autores são integralmente responsáveis pelo conteúdo (itens c, d, f).

## Referências

JOCHER, G.; CHAURASIA, A.; QIU, J. **Ultralytics YOLOv8**. 2023. Disponível em: https://docs.ultralytics.com/

REDMON, J.; DIVVALA, S.; GIRSHICK, R.; FARHADI, A. **You Only Look Once: Unified, Real-Time Object
Detection**. In: CVPR, 2016.

ZHANG, Z. **A Flexible New Technique for Camera Calibration**. IEEE Transactions on Pattern Analysis and
Machine Intelligence, v. 22, n. 11, 2000.

BROOKE, J. **SUS: A "quick and dirty" usability scale**. In: Usability Evaluation in Industry. Taylor &
Francis, 1996.

BRADSKI, G. **The OpenCV Library**. Dr. Dobb's Journal of Software Tools, 2000.

SZELISKI, R. **Computer Vision: Algorithms and Applications**. 2. ed. Springer, 2022.